In [19]:
# Load Packages
from datetime import date, datetime, timedelta
import calendar
import pandas as pd
import numpy as np
import itertools
import copy
from numpy.linalg import multi_dot
from scipy.stats import norm
import matplotlib.pyplot as plt
import math
import scipy.stats as st
import enum
import scipy.optimize as optimize

#Import Date opertative user defined functions
from ipynb.fs.full.user_defined_vik_functions import get_all_monthly_option_expiries, \
                                                     find_last_thurs_date_of_month, \
                                                     prev_workday_if_holiday, prev_weekday, \
                                                     find_wkly_expries,date_of_prev_thurs, \
                                                     generate_daily_dates_each_month

# Import dataframe naming functions 
from ipynb.fs.full.user_defined_vik_functions import get_mthly_df_name_from_expiry

#Import data loading functions
from ipynb.fs.full.user_defined_vik_functions import load_all_mthly_data

# Import functions of option Greeks
from ipynb.fs.full.user_defined_vik_functions import call_delta_with_spot_and_div, put_delta_with_spot_and_div

#Import implied volatility processing functions
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_iv, gen_interpolated_iv

#Import Risk-free interest rate generation functions
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_ir

# Import risk free interest rate function
from ipynb.fs.full.user_defined_vik_functions import get_risk_free_rate_from_exact_date




In [20]:
##########################################
#      Dynamic Hedging by Futures        #
#########################################

#Index of Interest
stock_ident = "BANKNIFTY"
# stock_ident = "NIFTY"

#Static hedging performed at different moneyness regions 
#i.e. moneyness is used to select the option with nearest moneyness match
# ATM - At the Money, ITM - In the money, OTM - Out of the Money
prod_moneyness = "ATM"

#Product type to hedge: either "CE" or "PE"
prod_type = "CE"
# prod_type = "PE"

#Path to refer data
source_path = "/home/jupyter-partha/Vikranth - Chapter 2/"
input_sub_path = "Input Data/mkt_data_covid_region/"
output_sub_path = "Output Data/"
input_data_path = source_path + input_sub_path
output_data_path = source_path + output_sub_path


# Periods of interest will be a dictionary
#Key is the year, value is a list of months 1-12, 1- Jan, 2 - Feb,...12 - Dec
# For E.g., periods_of_interest = {2020: [3], 2019: [11, 12]}
# periods_of_interest = {2019: [8, 9, 10, 11, 12], 2020: [1, 2, 3, 4, 5, 6, 7]}
periods_of_interest = {2020: [3]}

#List of holidays
holidays_list = [date(2019, 3, 4), date(2019, 3, 21),\
                 date(2019, 4, 17), date(2019, 4, 19), date(2019, 4, 29),\
                 date(2019, 5, 1),\
                 date(2019, 6, 5),\
                 date(2019, 8, 12), date(2019, 8, 15),\
                 date(2019, 9, 2), date(2019, 9, 10), \
                 date(2019, 10, 2), date(2019, 10, 8), date(2019, 10, 21), date(2019, 10, 28), \
                 date(2019, 11, 12), \
                 date(2019, 12, 25), \
                 date(2020, 2, 21), \
                 date(2020, 3, 10), \
                 date(2020, 4, 2), date(2020, 4, 6), date(2020, 4, 10), date(2020,4, 14), \
                 date(2020, 5, 1), date(2020, 5, 25), \
                 date(2020, 10, 2), date(2020, 11, 16), date(2020, 11, 30), date(2020, 12, 25)]


i = np.complex(0.0, 1.0)


In [21]:

def CallPutOptionPriceCOSMthd(cf, prod_type, S0, r, tau, K, N, L):
    #cf - Characteristic function as a function
    # prod_type: "CE" or "PE" - +1 for call and -1 for Put
    # S0 - Initial stock price
    # r -  Interest rate (constant)
    # tau - Time to Maturity
    # K-strike price
    # N - Number of Expansion terms
    # L - Size of truncation domain
    
    if (prod_type == "CE"):
        CP = 1
    else:
        CP = -1
    
    # Reshape K to a column vector
    if K is not np.array:
        K = np.array(K).reshape([len(K), 1])
    
    i = np.complex(0.0, 1.0)
    x0 = np.log(S0/K)
    
    # Truncation domain
    a = 0.0 - L * np.sqrt(tau)
    b = 0.0 + L * np.sqrt(tau)
    
    # Summation from k=0 to k=N-1
    
    k = np.linspace(0, N-1,N).reshape([N,1])
    u = k * np.pi / (b-a)
    
    # Find option coefficients
    
    H_k = CallPutCoefficients(CP, a, b, k)
    mat = np.exp(i * np.outer((x0 - a), u))
    temp = cf(u) * H_k
    temp[0] = 0.5 * temp[0]
    value = np.exp(-r*tau) * K * np.real(mat.dot(temp))
    
    return value

def CallPutCoefficients(prod_type, a, b, k):

    if (prod_type == "CE"):
        CP = 1
    else:
        CP = -1
     
    if prod_type == "CE":
        c=0.0
        d=b
        coef = Chi_Psi(a, b, c, d, k)
        Chi_k = coef["chi"]
        Psi_k = coef["psi"]
        if a<b and b<0.0:
            H_k = np.zeros([len(k), 1])
        else:
            H_k = 2.0 /(b-a) * (Chi_k - Psi_k)
        
    else:
        c = a
        d = 0.0
        coef = Chi_Psi(a,b,c,d,k)
        Chi_k = coef["chi"]
        Psi_k = coef["psi"]
        H_k = 2.0 / (b-a) * (-Chi_k + Psi_k)
        
    return H_k

def Chi_Psi(a, b, c, d, k):
    psi = np.sin(k * np.pi * (d-a) / (b-a)) - np.sin(k * np.pi * (c-a)/(b-a))
    psi[1:] = psi[1:] * (b-a) / (k[1:] * np.pi)
    psi[0] = d - c
    
    chi = 1.0 / (1.0 + np.power((k * np.pi / (b-a)) , 2))
    expr1 = np.cos(k * np.pi * (d-a)/(b-a)) * np.exp(d) - np.cos(k * np.pi * (c-a) / (b-a)) * np.exp(c)
    expr2 = k * np.pi / (b-a) * np.sin(k * np.pi * (d-a)/(b-a)) - k * np.pi / (b-a) * np.sin(k * np.pi * (c-a)/(b-a)) * np.exp(c) 
    chi = chi * (expr1 + expr2)
    value = {"chi": chi, "psi":psi}
    return value


def ChFHestonModel(r, tau, kappa, gamma, vbar, v0, rho):
    i = np.complex(0.0,1.0)
    D1 = lambda u: np.sqrt(np.power(kappa-gamma*rho*i*u,2) + (u*u + i*u) * gamma * gamma)
    g = lambda u: (kappa - gamma *rho*i*u - D1(u)) / (kappa-gamma*rho*i*u + D1(u))
    C = lambda u: (1.0 - np.exp(-D1(u)*tau)) / (gamma * gamma * (1.0 - g(u)*np.exp(-D1(u)*tau))) * (kappa - gamma*rho*i*u - D1(u))
    
    # Note that we exclude the term -r*tau, as the discounting is performed in the COS method
    A = lambda u: r*i*u*tau + kappa*vbar*tau/gamma/gamma  *  (kappa - gamma*rho*i*u-D1(u)) - 2*kappa*vbar/gamma/gamma *np.log((1.0 - g(u)*np.exp(-D1(u)*tau))/ (1.0 - g(u)))
    
    # Characteristic function for Heston Model
    cf = lambda u: np.exp(A(u) + C(u)*v0)
    
    return cf

  
if (prod_type == "CE"):
    CP = 1
else:
    CP = -1

S0 = 105.0
r = 0.05
tau = 3.0
K = np.array([90, 100, 105, 110])
K = np.array(K).reshape(-1, 1)
    
N = 5000 #fixed as of now
L = 5 #fixed as of now
    
#Heston Model Parameters
kappa = 0.1
gamma = 0.1 #vol of vol
vbar = 0.05 #long run variance
rho = -0.75
v0 = 0.05 #initial variance
#vol = np.sqrt(v0)
    
cf = ChFHestonModel(r, tau, kappa, gamma, vbar, v0, rho)
PV = CallPutOptionPriceCOSMthd(cf, CP, S0, r, tau, K, N, L)

print(PV)    

 

[[ 4.84834125]
 [ 7.36580614]
 [ 8.88698695]
 [10.58976012]]


In [24]:

#Dynamic Hedging Function
def heston_calibration(mthly_expiries_list, dict_wkly_expiries_each_month, mthly_mkt_data,\
                           prod_type, prod_moneyness, holidays_list, \
                           output_path, stock_ident):
        
    mth_list_of_df = []
    df_strikes_hedged = pd.read_csv(output_path + "B1A_LR_" + stock_ident + "_" + prod_moneyness + "_" + prod_type + "_strikes_hedged.csv")

    for each_month in mthly_expiries_list:
        strike = df_strikes_hedged[df_strikes_hedged['Date'] == each_month.strftime("%d-%b-%Y")]['Strike'].iloc[0]
        df_month = mthly_mkt_data[get_mthly_df_name_from_expiry(each_month, prod_type, holidays_list, stock_ident=stock_ident)]

# Remove the below two lines - if we need to use "Close price" instead of "Settlement Price"

        df_month['Date'] = df_month.apply(lambda x: x['Date'].split("-")[0] + "-" + \
                                  x['Date'].split("-")[1]  + "-"  + "20" + x['Date'].split("-")[2] \
                                  if (len(x['Date'].split("-")[2]) == 2) else x['Date'], axis=1) 
        df_month['Expiry'] = df_month.apply(lambda x: x['Expiry'].split("-")[0] + "-" + \
                                  x['Expiry'].split("-")[1]  + "-"  + "20" + x['Expiry'].split("-")[2] \
                                  if (len(x['Expiry'].split("-")[2]) == 2) else x['Expiry'], axis=1)

        if (prod_type == "CE"):
            df_month['Close'] = df_month.apply(lambda x: max(x['Underlying Value'] - x['Strike Price'], 0) if (x['Date'] == x['Expiry']) else \
                                                 x['Settle Price'], axis=1)
        else:
            df_month['Close'] = df_month.apply(lambda x: max(x['Strike Price']-x['Underlying Value'], 0) if (x['Date'] == x['Expiry']) else \
                                                 x['Settle Price'], axis=1)            
        df_month = df_month[['Date', 'Expiry', 'Underlying Value', 'Strike Price', 'Close']]
        df_month = df_month[df_month['Strike Price'] == strike]
        
        start_day = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][0].strftime("%d-%b-%Y")
        df_fut = mthly_mkt_data['df_FUTIDX_' + stock_ident + '_' + start_day + '_TO_' + each_month.strftime("%d-%b-%Y")]
        df_fut['Date'] = df_fut.apply(lambda x: x['Date'].split("-")[0] + "-" + \
                                  x['Date'].split("-")[1]  + "-"  + "20" + x['Date'].split("-")[2] \
                                  if (len(x['Date'].split("-")[2]) == 2) else x['Date'], axis=1) 
        df_fut['Expiry'] = df_fut.apply(lambda x: x['Expiry'].split("-")[0] + "-" + \
                                  x['Expiry'].split("-")[1]  + "-"  + "20" + x['Expiry'].split("-")[2] \
                                  if (len(x['Expiry'].split("-")[2]) == 2) else x['Expiry'], axis=1)        
        
        # Remove the following lines if required to use close price instead of settlement price
        df_fut['Close'] = df_fut['Settle Price']

        df_month = df_month.merge(df_fut, how ='left', left_on='Date', right_on='Date', suffixes=('', '_FUT'))
#         print(df_fut)
        df_month = df_month[['Date', 'Expiry', 'Underlying Value', 'Strike Price', 'Close', 'Close_FUT']]
        
            
        df_month['Date'] = df_month.apply(lambda x: x['Date'].split("-")[0] + "-" + \
                                                        x['Date'].split("-")[1]  + "-"  + \
                                                        "20" + x['Date'].split("-")[2] \
                                                        if (len(x['Date'].split("-")[2]) == 2) else x['Date'], axis=1)  
        
        df_month['Expiry'] = df_month.apply(lambda x: x['Expiry'].split("-")[0] + "-" + \
                                                        x['Expiry'].split("-")[1]  + "-"  + \
                                                        "20" + x['Expiry'].split("-")[2] \
                                                        if (len(x['Expiry'].split("-")[2]) == 2) else x['Expiry'], axis=1)  


        df_month["Moneyness"] = np.divide(df_month['Underlying Value'], df_month['Strike Price'])

        #Vol Surface Input
        df_full_iv = pd.read_csv(output_path + "A3_" + stock_ident + "_implied_vol_surface.csv")
        df_iv = df_full_iv[df_full_iv['Month_Expiry'] == each_month.strftime("%d-%b-%Y")]
        
        df_month_to_find_delta = copy.deepcopy(df_month) 

        df_month_to_find_delta = df_month_to_find_delta.assign(x=pd.to_datetime(df_month_to_find_delta['Date'])).sort_values('x').drop('x', 1).reset_index(drop=True)
        
        # This is to remove dates such as holidays as this is considered while generating iv
        unique_iv_dates = np.unique(np.array(df_iv.loc[:,'Date'])).reshape(-1)
        
        df_dates_iv_file = pd.DataFrame({"IV_Dates": unique_iv_dates, "Indicator": [1 for i in range(0, unique_iv_dates.shape[0])]})
        df_month_to_find_delta = pd.merge(df_month_to_find_delta, df_dates_iv_file, \
                                          how = 'left', left_on = ['Date'], right_on = ['IV_Dates'])
        df_month_to_find_delta = df_month_to_find_delta[df_month_to_find_delta["Indicator"] == 1]
#         print(df_month_to_find_delta)
        # Smile Vol
        df_month_to_find_delta['iv'] = df_month_to_find_delta.apply(lambda x: get_iv_linearIntpl(df_iv, x['Date'], x['Moneyness']) \
                                                                    if (datetime.strptime(x['Date'], "%d-%b-%Y").date() < each_month) else \
                                                                    0.0, axis=1)     
        df_month_to_find_delta['r_f'] = df_month_to_find_delta.apply(lambda x: get_risk_free_rate_from_exact_date(datetime.strptime(x['Date'], "%d-%b-%Y").date()), axis=1)   
        df_month_to_find_delta['expiry_dt'] = df_month_to_find_delta.apply(lambda x: float((each_month - datetime.strptime(x['Date'], "%d-%b-%Y").date()).days) / 365, axis=1)
        df_month_to_find_delta['r'] = df_month_to_find_delta.apply(lambda x: np.log(x['Close_FUT']/x['Underlying Value'])/x['expiry_dt'], axis=1)
        
        if (prod_type == "CE"):
            df_month_to_find_delta['delta'] = df_month_to_find_delta.apply(lambda x: \
                                              call_delta_with_spot_and_div(x['Underlying Value'], x['Strike Price'], \
                                                                   x['r'], x['r_f'], x['iv'], x['expiry_dt']) \
                                              if (datetime.strptime(x['Date'], "%d-%b-%Y").date() < each_month) else \
                                              x['Underlying Value'] * 0.0, axis=1)
        else:
            df_month_to_find_delta['delta'] = df_month_to_find_delta.apply(lambda x: \
                                              put_delta_with_spot_and_div(x['Underlying Value'], x['Strike Price'], \
                                                                   x['r'], x['r_f'], x['iv'], x['expiry_dt']) \
                                              if (datetime.strptime(x['Date'], "%d-%b-%Y").date() < each_month) else \
                                              x['Underlying Value'] * 0.0, axis=1)
        
        
        #Lag Variables
        df_month_to_find_delta['delta_lag'] = [0.0] + list(df_month_to_find_delta['delta'])[:-1]
        df_month_to_find_delta['underlying_lag'] = [0.0] + list(df_month_to_find_delta['Underlying Value'])[:-1]
        df_month_to_find_delta['Close_lag'] = [0.0] + list(df_month_to_find_delta['Close'])[:-1]
        df_month_to_find_delta['Date_lag'] = [start_day] + list(df_month_to_find_delta['Date'])[:-1]
        
        df_month_to_find_delta.loc[0, 'Close_lag'] = df_month_to_find_delta.loc[0, 'Close'] 
        df_month_to_find_delta.loc[0, 'underlying_lag'] = df_month_to_find_delta.loc[0, 'Underlying Value'] 
        df_month_to_find_delta.loc[0, 'Date_lag'] = df_month_to_find_delta.loc[0, 'Date'] 
        df_month_to_find_delta.loc[0, 'delta_lag'] = df_month_to_find_delta.loc[0, 'delta']
        
        df_month_to_find_delta['delta_lag'] = [0.0] + list(df_month_to_find_delta['delta'])[:-1]

        df_month_to_find_delta['hedge_dt'] = df_month_to_find_delta.apply(lambda x: float((datetime.strptime(x['Date'], "%d-%b-%Y").date() - \
                                                                                           datetime.strptime(x['Date_lag'], "%d-%b-%Y").date()).days) / 365, axis=1)
        
        
# We find the amount of stocks purchased to form the dynamic hedge pfl
# We need to find teh stocks bought as of previous to build dynamic pfl yesterday
        df_month_to_find_delta['new_stocks_purchase_qty'] = df_month_to_find_delta['delta'] - df_month_to_find_delta['delta_lag']
        df_month_to_find_delta['new_stocks_purchase_qty_lag'] = [0.0] + list(df_month_to_find_delta['new_stocks_purchase_qty'])[:-1]
        df_month_to_find_delta['abs_new_stocks_purchase_qty_lag'] = df_month_to_find_delta['new_stocks_purchase_qty_lag'].abs()
        equity_trans_cost = 0.0000345

#         df_month_hedge_error = df_month_to_find_delta[df_month_to_find_delta['Date'] != df_month_to_find_delta['Date'].iloc[0]]
        df_month_hedge_error = df_month_to_find_delta
        df_month_hedge_error['Hedge Error Date'] = df_month_hedge_error.loc[:, 'Date'] 
        
        df_month_hedge_error['Dynamic Hedge Pfl Value'] = df_month_to_find_delta.apply(lambda x: (x['Close_lag'] - x['abs_new_stocks_purchase_qty_lag'] * equity_trans_cost - \
                                                                                                    x['delta_lag'] * x['underlying_lag']) \
                                                                                                    * np.exp(x['r_f'] * x['hedge_dt']) + \
                                                                                                    x['delta_lag'] * x['Underlying Value'], \
                                                                                                    axis=1)
        df_month_hedge_error['Dynamic Hedge Error'] = df_month_hedge_error.apply(lambda x: x['Close'] - x['Dynamic Hedge Pfl Value'], axis=1)

        # The following piece is to filter data except last week as the long options expiry match weekly options
        last_week_start = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][-1]
        df_month_hedge_error["date_ind"] = df_month_hedge_error.apply(lambda x: 1 \
                                                                      if datetime.strptime(x['Hedge Error Date'], "%d-%b-%Y").date() <= last_week_start
                                                                      else 0, axis=1) 
        
        df_month_hedge_error = df_month_hedge_error[df_month_hedge_error["date_ind"] == 1]
        df_month_hedge_error = df_month_hedge_error[['Hedge Error Date', 'Expiry', 'iv', 'delta', 'delta_lag', 'Underlying Value', 'Strike Price', 'r', 'expiry_dt', 'Close', 'Dynamic Hedge Pfl Value', 'Dynamic Hedge Error']]
        df_month_hedge_error = df_month_hedge_error.rename(columns={"Hedge Error Date":"Date", "Expiry":"Month_Expiry"})
        mth_list_of_df.append(df_month_hedge_error)
        
    df_dynamic_hedge = pd.concat(mth_list_of_df, axis=0)

    df_dynamic_hedge.to_csv( output_path + "D1_" + stock_ident + "_" + prod_moneyness + "_" + prod_type + "_dynamic_hedge.csv", index = False)

    return df_dynamic_hedge
  

In [25]:
#Compute Daily Dynamic Hedging Errors

###################################
# Common Variables initialisation #
###################################

#Find the monthly strike of option from mkt data to find the option to be hedged
#Every month, an option is hedged
mthly_expiries_list = get_all_monthly_option_expiries(periods_of_interest, holidays_list)
dict_wkly_expiries_each_month = find_wkly_expries(mthly_expiries_list, holidays_list)

#Load all monthly mkt data
mthly_mkt_data = load_all_mthly_data(mthly_expiries_list, input_data_path, holidays_list, prod_type_lists=["FUT", "CE", "PE"], stock_ident=stock_ident)

#Load weekly risk free interest rates from futures
#This function to be changed based on updates on risk-free-interest 
dict_wkly_ir = generate_weekly_ir(dict_wkly_expiries_each_month, mthly_mkt_data, stock_ident)


df_dynamic_hedge = dynamic_hedge_pfl_error(mthly_expiries_list, dict_wkly_expiries_each_month, mthly_mkt_data,\
                                           prod_type, prod_moneyness, holidays_list, 
                                           output_data_path, stock_ident)


In [ ]:
import numpy as np
import pandas as pd
import enum
from datetime  import  date, datetime, timedelta 
import QuantLib as ql
import scipy
from scipy.optimize import root
from math import pow, sqrt
from scipy.optimize import differential_evolution
import time
import random
import math

class OptionType(enum.Enum):
    CALL = 1.0
    PUT = -1.0

i = np.complex(0.0, 1.0)


def CallPutOptionPriceCOSMthd(cf, prod_type, S0, r, tau, K, N, L):
    #cf - Characteristic function as a function
    # CP - +1 for call and -1 for Put
    # S0 - Initial stock price
    # r -  Interest rate (constant)
    # tau - Time to Maturity
    # K-strike price
    # N - Number of Expansion terms
    # L - Size of truncation domain
    
    
    # Reshape K to a column vector
    if K is not np.array:
        K = np.array(K).reshape([len(K), 1])
    
    i = np.complex(0.0, 1.0)
    x0 = np.log(S0/K)
    
    # Truncation domain
    a = 0.0 - L * np.sqrt(tau)
    b = 0.0 + L * np.sqrt(tau)
    
    # Summation from k=0 to k=N-1
    
    k = np.linspace(0, N-1,N).reshape([N,1])
    u = k * np.pi / (b-a)
    
    # Find option coefficients
    
    H_k = CallPutCoefficients(CP, a, b, k)
    mat = np.exp(i * np.outer((x0 - a), u))
    temp = cf(u) * H_k
    temp[0] = 0.5 * temp[0]
    value = np.exp(-r*tau) * K * np.real(mat.dot(temp))
    
    return value

def CallPutCoefficients(CP, a, b, k):
    if CP==OptionType.CALL:
        c=0.0
        d=b
        coef = Chi_Psi(a, b, c, d, k)
        Chi_k = coef["chi"]
        Psi_k = coef["psi"]
        if a<b and b<0.0:
            H_k = np.zeros([len(k), 1])
        else:
            H_k = 2.0 /(b-a) * (Chi_k - Psi_k)
        
    elif CP==OptionType.PUT:
        c = a
        d = 0.0
        coef = Chi_Psi(a,b,c,d,k)
        Chi_k = coef["chi"]
        Psi_k = coef["psi"]
        H_k = 2.0 / (b-a) * (-Chi_k + Psi_k)
        
    return H_k

def Chi_Psi(a, b, c, d, k):
    psi = np.sin(k * np.pi * (d-a) / (b-a)) - np.sin(k * np.pi * (c-a)/(b-a))
    psi[1:] = psi[1:] * (b-a) / (k[1:] * np.pi)
    psi[0] = d - c
    
    chi = 1.0 / (1.0 + np.power((k * np.pi / (b-a)) , 2))
    expr1 = np.cos(k * np.pi * (d-a)/(b-a)) * np.exp(d) - np.cos(k * np.pi * (c-a) / (b-a)) * np.exp(c)
    expr2 = k * np.pi / (b-a) * np.sin(k * np.pi * (d-a)/(b-a)) - k * np.pi / (b-a) * np.sin(k * np.pi * (c-a)/(b-a)) * np.exp(c) 
    chi = chi * (expr1 + expr2)
    value = {"chi": chi, "psi":psi}
    return value


def ChFHestonModel(r, tau, kappa, gamma, vbar, v0, rho):
    i = np.complex(0.0,1.0)
    D1 = lambda u: np.sqrt(np.power(kappa-gamma*rho*i*u,2) + (u*u + i*u) * gamma * gamma)
    g = lambda u: (kappa - gamma *rho*i*u - D1(u)) / (kappa-gamma*rho*i*u + D1(u))
    C = lambda u: (1.0 - np.exp(-D1(u)*tau)) / (gamma * gamma * (1.0 - g(u)*np.exp(-D1(u)*tau))) * (kappa - gamma*rho*i*u - D1(u))
    
    # Note that we exclude the term -r*tau, as the discounting is performed in the COS method
    A = lambda u: r*i*u*tau + kappa*vbar*tau/gamma/gamma  *  (kappa - gamma*rho*i*u-D1(u)) - 2*kappa*vbar/gamma/gamma *np.log((1.0 - g(u)*np.exp(-D1(u)*tau))/ (1.0 - g(u)))
    
    # Characteristic function for Heston Model
    cf = lambda u: np.exp(A(u) + C(u)*v0)
    
    return cf

def hestonPriceEY(CP, K, r, S0, tau, gamma, rho, vbar, v0, kappa=0.5, N=5000, L=5):    
    cf = ChFHestonModel(r, tau, kappa, gamma, vbar, v0, rho)
    heston_pv = CallPutOptionPriceCOSMthd(cf, CP, S0, r, tau, K, N, L)
    
    return heston_pv  

def obj_func(params_array, dict_mkt_value, dict_strikes, r, spot, tau, v0):
    mkt_call_price = np.array(dict_mkt_value["CE"]).reshape(-1,1)
    mkt_put_price = np.array(dict_mkt_value["PE"]).reshape(-1,1)
    call_strikes = dict_strikes["CE"]
    put_strikes = dict_strikes["PE"]
    
    rho = params_array[0]
    vbar = params_array[1]
    gamma = params_array[2]
    kappa = params_array[3]
   
    heston_call_price = hestonPriceEY(OptionType.CALL, call_strikes, r, spot, tau, gamma, rho, vbar, v0, kappa).reshape(-1,1)
    heston_put_price = hestonPriceEY(OptionType.PUT, put_strikes, r, spot, tau, gamma, rho, vbar, v0, kappa).reshape(-1,1)
   
    call_error = np.subtract(mkt_call_price, heston_call_price)
    put_error = np.subtract(mkt_put_price, heston_put_price)
    
    #obj_err_func = np.sum(call_error**2) + np.sum(put_error**2)
    obj_err_func = (np.sum(abs(call_error)) + np.sum(abs(put_error)))
    
     
    return obj_err_func


start_time = time.monotonic()

v0 = np.square(0.1901033861388114)

#Initialisation

vbar = v0 #Long term variance//
rho = -0.25 #Correlation between brownian motions (spot and variance process)
gamma = 0.1 #Vol of Vol
kappa = 0.5
    
params_array = np.array([rho, vbar, gamma, kappa]).reshape(-1)
    

bounds = [(-1,1), (0.0001,100),(0.0001,100),(0.0001,100)]

input_data_path=r'C:\Users\YQ938PF\Desktop\Python scripts\\'
filename_call="OPTIDX_BANKNIFTY_CE_27-Feb-2020_TO_26-Mar-2020.csv"
filename_put = "OPTIDX_BANKNIFTY_PE_27-Feb-2020_TO_26-Mar-2020.csv"


start_date=date(2020,2,27)
month=date(2020,3,26)
tau = float((month - start_date).days) / 365.0
r = 0.05

df_call = pd.read_csv(input_data_path + filename_call)
df_call = df_call[df_call["Date"] == '27-Feb-20']
df_call = df_call[df_call["Strike Price"] >  30187 ]
df_call = df_call[df_call["No. of contracts"] > 5]

df_put = pd.read_csv(input_data_path + filename_put)
df_put = df_put[df_put["Date"] == '27-Feb-20']
df_put = df_put[df_put["Strike Price"] <  30187]
df_put = df_put[df_put["No. of contracts"] > 5]
#spot = np.array(df_call["Underlying Value"]).reshape(-1,1)

#LENGTH OF CALL AND PUTS NEED TO BE DETERMENED, TO GET AVG ERROR;   
call_mkt_value = np.array(df_call["Close"]).reshape(-1)
put_mkt_value = np.array(df_put["Close"]).reshape(-1)
dict_mkt_value = {"CE": call_mkt_value, "PE": put_mkt_value}
    
    
call_strikes = np.array(df_call["Strike Price"]).reshape(-1,1)
put_strikes = np.array(df_put["Strike Price"]).reshape(-1,1)
dict_strikes = {"CE": call_strikes, "PE": put_strikes}
spot = np.array(df_call["Underlying Value"]).reshape(-1,1)
spot = np.unique(spot)

# from scipy.optimize import NonlinearConstraint
# def constraint(params_array):
#     return ((params_array[2]*params_array[2]) - (2*params_array[3]*params_array[1]))
# nlc = NonlinearConstraint(constraint, -np.inf, 0)

result_dv = scipy.optimize.differential_evolution(obj_func,bounds = bounds, 
                                                  args=(dict_mkt_value, dict_strikes, r, spot, tau, v0), 
                                                  strategy='best1bin', maxiter=100, popsize=100, tol=0.01,
                                                  mutation=(0.5, 1), recombination=0.7, seed=None, callback=None, 
                                                  disp=False, polish=True, init='latinhypercube', atol=0, 
                                                  updating='immediate', workers=1)

params_array = np.array(result_dv.x).reshape(-1) 
result_dv.fun/(len(call_strikes) + len(put_strikes))
np.square(0.1901033861388114)


rho = params_array[0]
vbar = params_array[1]
gamma = params_array[2]
kappa = params_array[3]

params_array

def heston_simulation(params_array, initial_values, tau, time_steps, paths):
       
    rho = params_array[0]
    vbar = params_array[1]
    gamma = params_array[2]
    kappa = params_array[3]
    
    v0 = initial_values[0]*np.ones(paths).reshape(-1,1)
    S0 = initial_values[1]*np.ones(paths).reshape(-1,1)
    type(S0)
    
    random.seed(1)
    Z1= np.random.randn(paths).reshape(-1,1)
    random.seed(2)
    Z2 = np.random.randn(paths).reshape(-1,1)
    Z3 = rho*Z1 +np.sqrt(1-np.square(rho))*Z2
    i=0 
    dS =np.zeros(paths).reshape(-1,1)
    dV = np.zeros(paths).reshape(-1,1)
    dT = (tau/time_steps)*np.ones(paths).reshape(-1,1)
    
    montecarlo_simn_stock = S0
    montecarlo_simn_vol = v0
    global S_new
    S_new = S0 + dS
    V_new = v0 + dV
   
    for i in range(1,time_steps+1):
         
            dS = r*S_new*dT + np.sqrt(V_new)*S_new*Z1*np.sqrt(dT)
            dV = kappa*(vbar - V_new) + gamma*np.sqrt(V_new)*np.sqrt(dT)*Z3
        
                       
            S_new = S0 + dS
            V_new = abs(v0 + dV)
            montecarlo_simn_stock = np.append(montecarlo_simn_stock,S_new,axis=1)
            montecarlo_simn_vol = np.append(montecarlo_simn_vol,V_new,axis=1)
            
                      

    Stock_Result = pd.DataFrame(montecarlo_simn_stock)
    Stock_Result.to_csv(r"C:\Users\YQ938PF\Desktop\Python scripts\montecarlo_Stock_simulation2.csv")
    Vol_result = pd.DataFrame(montecarlo_simn_vol)
    Vol_result.to_csv(r"C:\Users\YQ938PF\Desktop\Python scripts\montecarlo_Vol_simulation2.csv")

initial_values = [v0, spot]     
time_steps = 100
paths = 100

# def convergence_check(S_new,paths):
#     return np.average(S_new)



heston_simulation(params_array, initial_values, tau, time_steps, paths)

# convergence_check(S_new, paths)

# paths_list = [50,100,200,500,1000,1500,2000,2500,3000,3500,4000,4500,5000,6000,
#               7000,8000,9000,10000,15000,20000,25000,30000,50000,100000,150000,200000]
# S_new_avg = []

# for paths in paths_list:
#     heston_simulation(params_array, initial_values, tau, time_steps, paths)
#     S_new_avg.append(convergence_check(S_new, paths))
    
# convergence = {"paths":paths_list,"s_new_avg":S_new_avg}
# convergence_df = pd.DataFrame(convergence)
# convergence_df.to_csv(r"C:\Users\YQ938PF\Desktop\Python scripts\montecarlo_Stock_convergence.csv")

time_steps = 1
paths = 50000
heston_simulation(params_array, initial_values, tau, time_steps, paths)
def option_price_montecarlo(strike,paths):
    k = strike * np.ones(paths).reshape(-1,1)
    zero = 0 * np.ones(paths).reshape(-1,1)
    result = np.average(math.exp(-(r*tau)) * np.maximum(S_new - k , zero))
    return result

option_price_montecarlo(30200, paths)

call_strikes_sort = np.sort(call_strikes, axis=None)

call_opn_price_montecarlo = []
for strike in call_strikes_sort:
    call_opn_price_montecarlo.append(option_price_montecarlo(strike, paths))
    
call_opn_price_mc_dict = {'strike_call':call_strikes_sort,'option_price_MC':call_opn_price_montecarlo}
call_opn_price_mc_df =  pd.DataFrame(call_opn_price_mc_dict)
call_opn_price_mc_df.to_csv(r"C:\Users\YQ938PF\Desktop\Python scripts\call_option_price_mc2.csv")

from scipy.stats import norm

#Black -Shcoles formula for European call options
def BSM_call(s_t, k, r, vol, dt):
    d_1_c = (1 / (vol * (dt ** 0.5))) 
    d1_log =  (np.log(s_t/k) + ((r + (vol ** 2) / 2) * dt))
    d_1 = d_1_c * d1_log
    d_2 = d_1 - vol * (dt ** 0.5)
    return (s_t * norm.cdf(d_1) - k * np.exp(-r * dt) * norm.cdf(d_2))

def BSM_put(s_t, k, r, vol, dt):
    d_1_c = (1 / (vol * (dt ** 0.5))) 
    d1_log =  (np.log(s_t/k) + ((r + (vol ** 2) / 2) * dt))
    d_1 = d_1_c * d1_log
    d_2 = d_1 - vol * (dt ** 0.5)
    return (k * np.exp(-r*dt) * norm.cdf(-d_2) - s_t *  norm.cdf(-d_1))

BSM_call(30187,30300, 0.05,0.1901033861388114 , tau)
BSM_put(30187, 30100, 0.05,0.1901033861388114 , tau)


In [ ]:
import numpy as np
import pandas as pd
import enum
from datetime import datetime 
import QuantLib as ql
import scipy
from scipy.optimize import root
from math import pow, sqrt
from scipy.optimize import differential_evolution
from scipy.optimize import basinhopping


class OptionType(enum.Enum):
    CALL = 1.0
    PUT = -1.0

i = np.complex(0.0, 1.0)


def CallPutOptionPriceCOSMthd(cf, CP, S0, r, tau, K, N, L):
    if K is not np.array:
        K = np.array(K).reshape([len(K), 1])
    
    i = np.complex(0.0, 1.0)
    x0 = np.log(S0/K)
    
    # Truncation domain
    a = 0.0 - L * np.sqrt(tau)
    b = 0.0 + L * np.sqrt(tau)
    
    # Summation from k=0 to k=N-1
    
    k = np.linspace(0, N-1,N).reshape([N,1])
    u = k * np.pi / (b-a)
    
    # Find option coefficients
    
    H_k = CallPutCoefficients(CP, a, b, k)
    mat = np.exp(i * np.outer((x0 - a), u))
    temp = cf(u) * H_k
    temp[0] = 0.5 * temp[0]
    value = np.exp(-r*tau) * K * np.real(mat.dot(temp))
    
    return value

def CallPutCoefficients(CP, a, b, k):
    if CP==OptionType.CALL:
        c=0.0
        d=b
        coef = Chi_Psi(a, b, c, d, k)
        Chi_k = coef["chi"]
        Psi_k = coef["psi"]
        if a<b and b<0.0:
            H_k = np.zeros([len(k), 1])
        else:
            H_k = 2.0 /(b-a) * (Chi_k - Psi_k)
        
    elif CP==OptionType.PUT:
        c = a
        d = 0.0
        coef = Chi_Psi(a,b,c,d,k)
        Chi_k = coef["chi"]
        Psi_k = coef["psi"]
        H_k = 2.0 / (b-a) * (-Chi_k + Psi_k)
        
    return H_k

def Chi_Psi(a, b, c, d, k):
    psi = np.sin(k * np.pi * (d-a) / (b-a)) - np.sin(k * np.pi * (c-a)/(b-a))
    psi[1:] = psi[1:] * (b-a) / (k[1:] * np.pi)
    psi[0] = d - c
    
    chi = 1.0 / (1.0 + np.power((k * np.pi / (b-a)) , 2))
    expr1 = np.cos(k * np.pi * (d-a)/(b-a)) * np.exp(d) - np.cos(k * np.pi * (c-a) / (b-a)) * np.exp(c)
    expr2 = k * np.pi / (b-a) * np.sin(k * np.pi * (d-a)/(b-a)) - k * np.pi / (b-a) * np.sin(k * np.pi * (c-a)/(b-a)) * np.exp(c) 
    chi = chi * (expr1 + expr2)
    value = {"chi": chi, "psi":psi}
    return value


def ChFHestonModel(r, tau, kappa, gamma, vbar, v0, rho):
    i = np.complex(0.0,1.0)
    D1 = lambda u: np.sqrt(np.power(kappa-gamma*rho*i*u,2) + (u*u + i*u) * gamma * gamma)
    g = lambda u: (kappa - gamma *rho*i*u - D1(u)) / (kappa-gamma*rho*i*u + D1(u))
    C = lambda u: (1.0 - np.exp(-D1(u)*tau)) / (gamma * gamma * (1.0 - g(u)*np.exp(-D1(u)*tau))) * (kappa - gamma*rho*i*u - D1(u))
    
    # Note that we exclude the term -r*tau, as the discounting is performed in the COS method
    A = lambda u: r*i*u*tau + kappa*vbar*tau/gamma/gamma  *  (kappa - gamma*rho*i*u-D1(u)) - 2*kappa*vbar/gamma/gamma *np.log((1.0 - g(u)*np.exp(-D1(u)*tau))/ (1.0 - g(u)))
    
    # Characteristic function for Heston Model
    cf = lambda u: np.exp(A(u) + C(u)*v0)
    
    return cf
# Squared error objective function
def obj_squared_error_func(params_array, dict_mkt_value, dict_strikes, r, spot, tau):
    mkt_call_price = np.array(dict_mkt_value["CE"]).reshape(-1,1)
    mkt_put_price = np.array(dict_mkt_value["PE"]).reshape(-1,1)
    call_strikes = dict_strikes["CE"]
    put_strikes = dict_strikes["PE"]
    gamma = params_array[0]
    rho = params_array[1]
    vbar = params_array[2]
    v0 = params_array[3]
    kappa = params_array[4]
    
    N=5000
    L=5
    cf = ChFHestonModel(r, tau, kappa, gamma, vbar, v0, rho)
    
    heston_call_price = CallPutOptionPriceCOSMthd(cf, OptionType.CALL, spot, r, tau, call_strikes, N, L)
    #heston_call_price = hestonPriceEY(OptionType.CALL, call_strikes, r, spot, tau, gamma, rho, vbar, v0, kappa).reshape(-1)
    heston_put_price = CallPutOptionPriceCOSMthd(cf,OptionType.PUT, spot, r, tau, put_strikes, N, L)

    #print(heston_call_price)
    #print(mkt_call_price)
    
    
    call_error = np.subtract(mkt_call_price, heston_call_price)
    put_error = np.subtract(mkt_put_price, heston_put_price)
   # obj_err_func = 0.5 * (np.dot(call_error, call_error)+np.dot(put_error, put_error))
    obj_err_func = np.sum(call_error**2) + np.sum(put_error**2)
    return obj_err_func

# absolute error objective function
def obj_func(params_array, dict_mkt_value, dict_strikes, r, spot, tau):
    mkt_call_price = np.array(dict_mkt_value["CE"]).reshape(-1,1)
    mkt_put_price = np.array(dict_mkt_value["PE"]).reshape(-1,1)
    call_strikes = dict_strikes["CE"]
    put_strikes = dict_strikes["PE"]
    gamma = params_array[0]
    rho = params_array[1]
    vbar = params_array[2]
    v0 = params_array[3]
    kappa = params_array[4]
    N=5000
    L=5
    cf = ChFHestonModel(r, tau, kappa, gamma, vbar, v0, rho)
    
    heston_call_price = CallPutOptionPriceCOSMthd(cf, OptionType.CALL, spot, r, 
                                                  tau, call_strikes, N, L)
    #heston_call_price = hestonPriceEY(OptionType.CALL, call_strikes, r, spot, tau, gamma, rho, vbar, v0, kappa).reshape(-1)
    heston_put_price = CallPutOptionPriceCOSMthd(cf,OptionType.PUT, spot, r, 
                                                 tau, put_strikes, N, L)
    call_error = np.absolute(np.subtract(mkt_call_price, heston_call_price)) 
    put_error = np.absolute(np.subtract(mkt_put_price, heston_put_price))
    total_error = np.sum(call_error) + np.sum(put_error)
    obj_err_func =  float(total_error)
#     print(call_error)
#     print(put_error)
    #print(params_array)
    return obj_err_func


#Initialisation
v0 = np.square(0.1901033861388114) #atm_call_ivInitial variance. We will use square of ATM volatility for initial variance
vbar = v0 #Long term variance//
rho = -0.25 #Correlation between brownian motions (spot and variance process)
gamma = 0.1 #Vol of Vol
kappa = 0.3
    
params_array = np.array([gamma, rho, vbar, v0, kappa]).reshape(-1)
    
#    heston_call_price = hestonPriceEY(OptionType.CALL, call_strikes, r, spot, tau, gamma, rho, vbar, v0, kappa).reshape(-1)
#    print(heston_call_price)

#    obj_squared_error_func(dict_mkt_value, dict_strikes, r, spot, tau, kappa, v0, gamma, rho, vbar)
#    gamma > 0, rho (-1,1), vbar > 0
bounds = [(0.000001,10), (-1,1), (0.000001,10),(0.000001,10),(0.000001,10)]

input_data_path=r'C:\Users\YQ938PF\Desktop\Python scripts\\'

filename_call="OPTIDX_BANKNIFTY_CE_27-Feb-2020_TO_26-Mar-2020.csv"
df_call = pd.read_csv(input_data_path + filename_call)

from datetime  import  date, datetime, timedelta
start_date=date(2020,2,27)
month=date(2020,3,26)

tau = float((month - start_date).days) / 365.0
r = 0.05


df_call = df_call[df_call["Date"] == '27-Feb-20']
df_call = df_call[df_call["Strike Price"] >  30187 ]
df_call = df_call[df_call["No. of contracts"] > 5]

#     print(df_call.head)
#data read put also   
filename_put = "OPTIDX_BANKNIFTY_PE_27-Feb-2020_TO_26-Mar-2020.csv"
df_put = pd.read_csv(input_data_path + filename_put)
df_put = df_put[df_put["Date"] == '27-Feb-20']
df_put = df_put[df_put["Strike Price"] <  30187]
df_put = df_put[df_put["No. of contracts"] > 5]
#spot = np.array(df_call["Underlying Value"]).reshape(-1,1)

#LENGTH OF CALL AND PUTS NEED TO BE DETERMENED, TO GET AVG ERROR;   
call_mkt_value = np.array(df_call["Close"]).reshape(-1)
put_mkt_value = np.array(df_put["Close"]).reshape(-1)
dict_mkt_value = {"CE": call_mkt_value, "PE": put_mkt_value}
    
    
call_strikes = np.array(df_call["Strike Price"]).reshape(-1,1)
put_strikes = np.array(df_put["Strike Price"]).reshape(-1,1)
dict_strikes = {"CE": call_strikes, "PE": put_strikes}
spot = np.array(df_call["Underlying Value"]).reshape(-1,1)
spot = np.unique(spot)

result_dv = scipy.optimize.differential_evolution(obj_func,bounds = bounds, args=(dict_mkt_value, dict_strikes, r, spot, tau), strategy='best1bin', maxiter=100, popsize=100, tol=0.01,mutation=(0.5, 1), recombination=0.7, seed=None, callback=None, disp=False, polish=True, init='latinhypercube', atol=0, updating='immediate', workers=1)

#calibrate
#match at the latest market;
#backtesting- barrier,exotics, mtm backtests;
#ml


print('DV:',  result_dv.fun)  
print('DV:', result_dv.x)    

dict_min = {"method": "L-BFGS-B", "args": (dict_mkt_value, dict_strikes, r, spot, 
                                           tau)}
class MyBounds:
    def __init__(self, xmax=[10.0, 1.0,9.0,8.0,7.0], xmin=[0.00001, -1.0, 0.0001, 0.001, 0.01] ):
        self.xmax = np.array(xmax)
        self.xmin = np.array(xmin)
    def __call__(self, **kwargs):
        x = kwargs["x_new"]
        tmax = bool(np.all(x <= self.xmax))
        tmin = bool(np.all(x >= self.xmin))
        return tmax and tmin

mybounds = MyBounds()
result_basin = scipy.optimize.basinhopping(obj_func, x0=params_array,
                                           niter=50, T=1.0, stepsize=0.5,
                                           minimizer_kwargs=dict_min, 
                                           accept_test=mybounds)
print('BH:', result_basin.fun)
print('BH:', result_basin.x)



In [ ]:
#!/usr/bin/env python
# coding: utf-8

# In[117]:


import numpy as np
from datetime import date, datetime, timedelta
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import enum
import scipy


#Import Date opertative user defined functions
from ipynb.fs.full.user_defined_vik_functions import get_all_monthly_option_expiries,                                                      find_last_thurs_date_of_month,                                                      prev_workday_if_holiday, find_wkly_expries,                                                     date_of_prev_thurs

# Import dataframe naming functions 
from ipynb.fs.full.user_defined_vik_functions import get_mthly_df_name_from_expiry, get_mthly_file_name_from_expiry

#Import data loading functions
from ipynb.fs.full.user_defined_vik_functions import load_all_mthly_data

#import traded options parameters and info
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_strikes, generate_mthly_strikes

#Import implied volatility processing functions
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_iv, gen_interpolated_iv, gen_bivariate_interpolated_iv

from ipynb.fs.full.A1_BS_Generate_daily_Implied_Vol_surface import gen_brent_iv_surface


# In[118]:


class OptionType(enum.Enum):
    CALL = 1.0
    PUT = -1.0

i = np.complex(0.0, 1.0)


# In[119]:


def CallPutOptionPriceCOSMthd(cf, CP, S0, r, tau, K, N=5000, L=5):
    #cf - Characteristic function as a function
    # CP - +1 for call and -1 for Put
    # S0 - Initial stock price
    # r -  Interest rate (constant)
    # tau - Time to Maturity
    # N - Number of Expansion terms
    # L - Size of truncation domain
    
    
    # Reshape K to a column vector
    if K is not np.array:
        K = np.array(K).reshape(-1, 1)
    
    i = np.complex(0.0, 1.0)
    x0 = np.log(S0/K)
    
    # Truncation domain
    a = 0.0 - L * np.sqrt(tau)
    b = 0.0 + L * np.sqrt(tau)
    
    # Summation from k=0 to k=N-1
    
    k = np.linspace(0, N-1,N).reshape([N,1])
    u = k * np.pi / (b-a)
    
    # Find option coefficients
    
    H_k = CallPutCoefficients(CP, a, b, k)
    mat = np.exp(i * np.outer((x0 - a), u))
    temp = cf(u) * H_k
    temp[0] = 0.5 * temp[0]
    value = np.exp(-r*tau) * K * np.real(mat.dot(temp))
    
    return value

def CallPutCoefficients(CP, a, b, k):
    if CP==OptionType.CALL:
        c=0.0
        d=b
        coef = Chi_Psi(a, b, c, d, k)
        Chi_k = coef["chi"]
        Psi_k = coef["psi"]
        if a<b and b<0.0:
            H_k = np.zeros([len(k), 1])
        else:
            H_k = 2.0 /(b-a) * (Chi_k - Psi_k)
        
    elif CP==OptionType.PUT:
        c = a
        d = 0.0
        coef = Chi_Psi(a,b,c,d,k)
        Chi_k = coef["chi"]
        Psi_k = coef["psi"]
        H_k = 2.0 / (b-a) * (-Chi_k + Psi_k)
        
    return H_k

def Chi_Psi(a, b, c, d, k):
    psi = np.sin(k * np.pi * (d-a) / (b-a)) - np.sin(k * np.pi * (c-a)/(b-a))
    psi[1:] = psi[1:] * (b-a) / (k[1:] * np.pi)
    psi[0] = d - c
    
    chi = 1.0 / (1.0 + np.power((k * np.pi / (b-a)) , 2))
    expr1 = np.cos(k * np.pi * (d-a)/(b-a)) * np.exp(d) - np.cos(k * np.pi * (c-a) / (b-a)) * np.exp(c)
    expr2 = k * np.pi / (b-a) * np.sin(k * np.pi * (d-a)/(b-a)) - k * np.pi / (b-a) * np.sin(k * np.pi * (c-a)/(b-a)) * np.exp(c) 
    chi = chi * (expr1 + expr2)
    value = {"chi": chi, "psi":psi}
    return value


def ChFHestonModel(r, tau, kappa, gamma, vbar, v0, rho):
    i = np.complex(0.0,1.0)
    D1 = lambda u: np.sqrt(np.power(kappa-gamma*rho*i*u,2) + (u*u + i*u) * gamma * gamma)
    g = lambda u: (kappa - gamma *rho*i*u - D1(u)) / (kappa-gamma*rho*i*u + D1(u))
    C = lambda u: (1.0 - np.exp(-D1(u)*tau)) / (gamma * gamma * (1.0 - g(u)*np.exp(-D1(u)*tau))) * (kappa - gamma*rho*i*u - D1(u))
    
    # Note that we exclude the term -r*tau, as the discounting is performed in the COS method
    A = lambda u: r*i*u*tau + kappa*vbar*tau/gamma/gamma  *  (kappa - gamma*rho*i*u-D1(u)) - 2*kappa*vbar/gamma/gamma *np.log((1.0 - g(u)*np.exp(-D1(u)*tau))/ (1.0 - g(u)))
    
    # Characteristic function for Heston Model
    cf = lambda u: np.exp(A(u) + C(u)*v0)
    
    return cf


def hestonPriceEY(CP, K, r, S0, tau, gamma, rho, vbar, v0, kappa=0.5, N=5000, L=5):
    
    cf = ChFHestonModel(r, tau, kappa, gamma, vbar, v0, rho)
    heston_pv = CallPutOptionPriceCOSMthd(cf, CP, S0, r, tau, K, N, L)
    
    return heston_pv    


# In[ ]:





# In[97]:


#########################################################################
#    Heston pricing of Europeans - COS Method Implementation Testing    #  
#########################################################################


#Trade Information
CP = OptionType.CALL
tau = float(1.0) # Time to Maturity of the contract
K = np.array([95.0, 100.0, 105.0]) #Strike


# Market Data
S0 = 100.0 #Spot
r = 0.05 #Risk-interest rate
atm_implied_vol = {"1W": np.sqrt(0.05), "2W": np.sqrt(0.05), "3W": np.sqrt(0.05), "1M": np.sqrt(0.05)}


#Heston params
kappa = 0.5 # We are fixing kappa to 0.5 for calibrations


heston_pv_EY = hestonPriceEY(CP, K, r, S0, tau, gamma, rho, vbar, v0, kappa=0.5, N=5000, L=5).reshape(-1)
# print(heston_pv_EY)
print(len(heston_pv_EY))

def obj_squared_error_func(params_array, dict_mkt_value, dict_strikes, r, spot, tau, kappa, v0):
    mkt_call_price = np.array(dict_mkt_value["CE"]).reshape(-1)
    mkt_put_price = np.array(dict_mkt_value["PE"]).reshape(-1)
    call_strikes = dict_strikes["CE"]
    put_strikes = dict_strikes["PE"]
    gamma = params_array[0]
    rho = params_array[1]
    vbar = params_array[2]
    heston_call_price = hestonPriceEY(OptionType.CALL, call_strikes, r, spot, tau, gamma, rho, vbar, v0, kappa).reshape(-1)
    heston_put_price = hestonPriceEY(OptionType.PUT, put_strikes, r, spot, tau, gamma, rho, vbar, v0, kappa).reshape(-1)
    call_error = np.subtract(mkt_call_price, heston_call_price) 
    put_error = np.subtract(mkt_put_price, heston_put_price)
    abs_err = np.sum(np.absolute(call_error)) + np.sum(np.absolute(put_error))
    mean_abs_err = float(abs_err)/ float(len(heston_call_price) + len(heston_put_price))
    print(mean_abs_err)
    obj_err_func = 0.5 * (np.dot(call_error, call_error) + np.dot(put_error, put_error))
    return obj_err_func

def obj_func(params_array, dict_mkt_value, dict_strikes, r, spot, tau, kappa, v0):
    mkt_call_price = np.array(dict_mkt_value["CE"]).reshape(-1)
    mkt_put_price = np.array(dict_mkt_value["PE"]).reshape(-1)
    call_strikes = dict_strikes["CE"]
    put_strikes = dict_strikes["PE"]
    gamma = params_array[0]
    rho = params_array[1]
    vbar = params_array[2]
    heston_call_price = hestonPriceEY(OptionType.CALL, call_strikes, r, spot, tau, gamma, rho, vbar, v0, kappa).reshape(-1)
    heston_put_price = hestonPriceEY(OptionType.PUT, put_strikes, r, spot, tau, gamma, rho, vbar, v0, kappa).reshape(-1)
    call_error = np.absolute(np.subtract(mkt_call_price, heston_call_price)) 
    put_error = np.absolute(np.subtract(mkt_put_price, heston_put_price))
    total_error = np.sum(call_error) + np.sum(put_error)
    obj_err_func =  float(total_error)/ float(len(heston_call_price) + len(heston_put_price))
#     print(call_error)
#     print(put_error)
    print(params_array)
    return obj_err_func


# In[128]:


#########################################################################
#  Calibration of Heston model - using Europeans pricing by COS method  #  
#########################################################################

# Stock Index of Interest
stock_ident = "BANKNIFTY"

# The scope of this code is to hedge one option 
# and scope will be extended to a portfolio
no_of_assets = 1
cor_mat = [[1]]

#Path to refer data
source_path = "/home/jupyter-partha/Vikranth - Chapter 2/"
input_sub_path = "Input Data/mkt_data_covid_region/"
output_sub_path = "Output Data/"
input_data_path = source_path + input_sub_path
output_data_path = source_path + output_sub_path

# Periods of interest will be a dictionary
#Key is the year, value is a list of months 1-12, 1- Jan, 2 - Feb,...12 - Dec
# For E.g., periods_of_interest = {2020: [3], 2019: [11, 12]}
# periods_of_interest = {2019: [11, 12], 2020: [1, 2, 3]}
periods_of_interest = {2020: [3]}

#List of holidays
holidays_list = [date(2019, 10, 28), date(2019, 11, 12), date(2019, 12, 25), date(2020, 2, 21), date(2020, 3, 10)]

#Find the monthly strikes of option from mkt data to find the option to be hedged
#Every month, an option is hedged
mthly_expiries_list = get_all_monthly_option_expiries(periods_of_interest, holidays_list)
dict_wkly_expiries_each_month = find_wkly_expries(mthly_expiries_list, holidays_list)

#Load weekly strikes of short term options used for hedging monthly options 
dict_ce_wkly_strikes, dict_pe_wkly_strikes, dict_wkly_spots = generate_weekly_strikes(dict_wkly_expiries_each_month, input_data_path, stock_ident = "BANKNIFTY")     

#Load all monthly mkt data
mthly_mkt_data = load_all_mthly_data(mthly_expiries_list, input_data_path, holidays_list, prod_type_lists=["FUT", "CE", "PE"], stock_ident=stock_ident)

class MyBounds:
    def __init__(self, xmax=[10.0, 1.0, 10.0], xmin=[0.00001, -1.0, 0.00001] ):
        self.xmax = np.array(xmax)
        self.xmin = np.array(xmin)
    def __call__(self, **kwargs):
        x = kwargs["x_new"]
        tmax = bool(np.all(x <= self.xmax))
        tmin = bool(np.all(x >= self.xmin))
        return tmax and tmin

for month in mthly_expiries_list:
    filename_call = get_mthly_file_name_from_expiry(month, "CE", holidays_list, stock_ident = stock_ident)
    filename_put = get_mthly_file_name_from_expiry(month, "PE", holidays_list, stock_ident = stock_ident)
    
    spot = dict_wkly_spots[month.strftime("%d-%b-%Y")][0]
    start_date = dict_wkly_expiries_each_month[month.strftime("%d-%b-%Y")][0]
    
    df_call = pd.read_csv(input_data_path + filename_call)
    df_call = df_call[df_call["Date"] == start_date.strftime("%d-%b-%Y")]
    df_call = df_call[df_call["Strike Price"] > spot]
    df_call = df_call[df_call["No. of contracts"] > 5]
    df_call.sort_values(by=["Strike Price"], inplace=True)
    
    df_put = pd.read_csv(input_data_path + filename_put)
    df_put = df_put[df_put["Date"] == start_date.strftime("%d-%b-%Y")]
    df_put = df_put[df_put["Strike Price"] < spot]
    df_put = df_put[df_put["No. of contracts"] > 5]
    df_put.sort_values(by=["Strike Price"], inplace=True)
    
    call_mkt_value = np.array(df_call["Close"]).reshape(-1)
    put_mkt_value = np.array(df_put["Close"]).reshape(-1)
    dict_mkt_value = {"CE": call_mkt_value, "PE": put_mkt_value}
    
    call_strikes = np.array(df_call["Strike Price"]).reshape(-1,1)
    put_strikes = np.array(df_put["Strike Price"]).reshape(-1,1)
    dict_strikes = {"CE": call_strikes, "PE": put_strikes}
    
    #Fixed Heston elements
    tau = float((month - start_date).days) / 365.0
    kappa = 0.5
    r = 0.05
    df_daily_call_iv_data = pd.read_csv(output_data_path + "A1_" + stock_ident + "_" + "Daily_vol_smile_" + "CE" + "_" + month.strftime("%d-%b-%Y") + ".csv")
    atm_call_iv = gen_bivariate_interpolated_iv(df_daily_call_iv_data, start_date.strftime("%d-%b-%Y"), 1.0, tau)

    df_daily_put_iv_data = pd.read_csv(output_data_path + "A1_" + stock_ident + "_" + "Daily_vol_smile_" + "PE" + "_" + month.strftime("%d-%b-%Y") + ".csv")
    atm_put_iv = gen_bivariate_interpolated_iv(df_daily_put_iv_data, start_date.strftime("%d-%b-%Y"), 1.0, tau)

    print(atm_call_iv, atm_put_iv)
    
    #################
    # 0.1901033861388114, 0.18997453768365324
    ################
    
    #Initialisation
    v0 = np.square((atm_call_iv + atm_put_iv)/2) #Initial variance. We will use square of ATM volatility for initial variance
    gamma = 0.81019177 
    rho = -0.50000834  
    vbar = 0.16158648
    heston_pv_EY_bas = hestonPriceEY(OptionType.CALL, call_strikes, r, spot, tau, gamma, rho, vbar, v0, kappa=0.5, N=5000, L=5).reshape(-1)
    heston_pv_put_EY_bas = hestonPriceEY(OptionType.PUT, put_strikes, r, spot, tau, gamma, rho, vbar, v0, kappa=0.5, N=5000, L=5).reshape(-1)

    gamma = 0.82033849 
    rho = -0.46871735  
    vbar = 0.13707478
      
    heston_pv_EY = hestonPriceEY(OptionType.CALL, call_strikes, r, spot, tau, gamma, rho, vbar, v0, kappa=0.5, N=5000, L=5).reshape(-1)
    heston_pv_put_EY = hestonPriceEY(OptionType.PUT, put_strikes, r, spot, tau, gamma, rho, vbar, v0, kappa=0.5, N=5000, L=5).reshape(-1)

    gamma = 0.69015544
    rho = -0.43589501
    vbar = 0.0456604
    
    heston_pv_call_vc = hestonPriceEY(OptionType.CALL, call_strikes, r, spot, tau, gamma, rho, vbar, v0, kappa=0.5, N=5000, L=5).reshape(-1)

    gamma = 0.81003372
    rho = -0.499996905 
    vbar = 0.16030929
    
    heston_pv_call_sj = hestonPriceEY(OptionType.CALL, call_strikes, r, spot, tau, gamma, rho, vbar, v0, kappa=0.5, N=5000, L=5).reshape(-1)
    heston_pv_put_sj = hestonPriceEY(OptionType.PUT, put_strikes, r, spot, tau, gamma, rho, vbar, v0, kappa=0.5, N=5000, L=5).reshape(-1)

    
#     print(spot)
#     print(call_strikes.reshape(-1))
#     print(heston_pv_EY)
#     print(call_mkt_value)
    dict_res = {"Strike": call_strikes.reshape(-1), "MV": call_mkt_value, "PV_dif_ev": heston_pv_EY, "PV_dif_ev_basin": heston_pv_EY_bas, "VC_results": heston_pv_call_vc, "SJ_results": heston_pv_call_sj}
    df_call_res = pd.DataFrame(dict_res)
    print(df_call_res)

    dict_res = {"Strike": put_strikes.reshape(-1), "MV": put_mkt_value, "PV_dif_ev": heston_pv_put_EY, "PV_dif_ev_basin": heston_pv_put_EY_bas, "SJ_results": heston_pv_put_sj}
    df_put_res = pd.DataFrame(dict_res)
    print(df_put_res)
    
    dv_call_error = np.sum(np.absolute(np.subtract(heston_pv_EY, call_mkt_value)))/25
    dv_put_error = np.sum(np.absolute(np.subtract(heston_pv_put_EY, put_mkt_value)))/16
    
    bas_call_error = np.sum(np.absolute(np.subtract(heston_pv_EY_bas, call_mkt_value)))/25
    bas_put_error = np.sum(np.absolute(np.subtract(heston_pv_put_EY_bas, put_mkt_value)))/16
    
    print(dv_call_error, dv_put_error)
    print(bas_call_error, bas_put_error)
    
#     vbar = v0 #Long term variance
#     rho = -0.25 #Correlation between brownian motions (spot and variance process)
#     gamma = 0.1 #Vol of Vol 
#     params_array = np.array([gamma, rho, vbar]).reshape(-1)
#    

############################# 
#     Heston Calibration    #
############################

# obj_func(dict_mkt_value, dict_strikes, r, spot, tau, kappa, v0, gamma, rho, vbar)

#     bounds = [(0.000001,10), (-1,1), (0.000001,10)]
#     result = scipy.optimize.differential_evolution(obj_func, \
#                                           bounds = bounds, \
#                                           args=(dict_mkt_value, dict_strikes, r, spot, tau, kappa, v0), \
#                                           strategy='best1bin', \
#                                           maxiter=1000, popsize=100, tol=0.01, mutation=0.5, \
#                                           recombination=0.7, seed=None, \
#                                           callback=None, disp=False, polish=True, \
#                                           init='latinhypercube', atol=0, updating='immediate', \
#                                           workers=1)
#     print(result.fun)
#     print(result.x)
    
    params_array = np.array(result.x).reshape(-1)
    gamma = params_array[0]
    rho = params_array[1]
    vbar = params_array[2]
    print(params_array)

#     dict_min = {"method": "L-BFGS-B", "args": (dict_mkt_value, dict_strikes, r, spot, tau, kappa, v0)}
#     bounds = [(0.000001,10), (-1,1), (0.000001,10)]    

    
#     scipy.optimize.basinhopping(func, x0, niter=100, T=1.0, stepsize=0.5, \
#                                 minimizer_kwargs=None, \
#                                 take_step=None, accept_test=None, callback=None, \
#                                 interval=50, disp=False, niter_success=None, seed=None)
    
#     mybounds = MyBounds()
#     result_basin = scipy.optimize.basinhopping(obj_squared_error_func, x0=params_array, niter=1000, T=1.0, stepsize=0.5, \
#                                 minimizer_kwargs=dict_min, accept_test=mybounds)

#     print(result_basin.fun)
    print(result_basin.x)
    print(spot)
    


# In[93]:


np.sqrt(3704 * 2) / 41
0.81019177 -0.50000834  0.16158648
# len(call_mkt_value) + len(put_mkt_value)

[0.69015544 -0.43589501 0.0456604]
0.81003372,-0.499996905,0.16030929



# In[127]:


#Function to generate Implied volatility Surface based on Brent's Algorithm
#Note: In vol surface: x-axis is moneyness ie. S/K for call options and K/S for put options
#                      y-axis is Time to maturity
#                      z-axis is implied volatility
#Inputs:
#       dt_list = [tenor1, tenor2....]
#       price_lists_tenorwise = [[Price for all options in tenor 1],[for tenor2],....]
#       strike_lists_tenorwise = [[Strike for all options in tenor 1],[for tenor2],....]
#       s_t - current stock price, r - risk free rate, option_type is "CE" or "PE"
#      initialisation of brent's iv is: [0.0000000001, 50, 50], 
#                 S.T, iv[0] creates a +ve error and iv[1] creates negative error,
#                where, error = (model price - mkt_price)
#Output:
#      A list of iv surface: [dict for tenor 1, dict for tenor 2,...]
#                      dict = {"Moneyness:" [], "T": [list of same value dt], "Impl_Vol": []}
# impl_vol_mkt = gen_brent_iv_surface([list(call_mkt_value)], spot, [list(call_strikes)], 0.05, [tau], "CE", [0.0000000001, 50, 50])
# impl_vol_heston = gen_brent_iv_surface([list(heston_pv_call_sj)], spot, [list(call_strikes)], 0.05, [tau], "CE", [0.0000000001, 50, 50])
print(pd.DataFrame(impl_vol_mkt[0]))
print(pd.DataFrame(impl_vol_heston[0]))
# print(impl_vol_mkt["Impl_Vol"])


# In[ ]:


#Function to generate Implied volatility Surface based on Brent's Algorithm
#Note: In vol surface: x-axis is moneyness ie. S/K for call options and K/S for put options
#                      y-axis is Time to maturity
#                      z-axis is implied volatility
#Inputs:
#       dt_list = [tenor1, tenor2....]
#       price_lists_tenorwise = [[Price for all options in tenor 1],[for tenor2],....]
#       strike_lists_tenorwise = [[Strike for all options in tenor 1],[for tenor2],....]
#       s_t - current stock price, r - risk free rate, option_type is "CE" or "PE"
#      initialisation of brent's iv is: [0.0000000001, 50, 50], 
#                 S.T, iv[0] creates a +ve error and iv[1] creates negative error,
#                where, error = (model price - mkt_price)
#Output:
#      A list of iv surface: [dict for tenor 1, dict for tenor 2,...]
#                      dict = {"Moneyness:" [], "T": [list of same value dt], "Impl_Vol": []}
def gen_brent_iv_surface(price_lists_tenorwise, s_t, strike_lists_tenorwise, r, dt_list, option_type, iv_list=[0.0000000001, 50, 50]):
    iv_surface = []
    for i in range(0, len(dt_list)):
        dt = dt_list[i]
        strike_list = strike_lists_tenorwise[i]
        price_list = price_lists_tenorwise[i]
        if(option_type == "CE"):
            moneyness_list = [s_t/strike for strike in strike_list]
        else:
            moneyness_list = [strike/s_t for strike in strike_list]
        dt_list_curr_tenor = [dt for evry_elm in price_list]
        spot_list = [s_t for evry_elm in price_list]
        impl_vol_list = [gen_brents_iv(price_list[i], s_t, strike_list[i], r, dt, option_type, iv_list) for i in range(0, len(price_list))]
        
        vol_smile_dict = {"Moneyness":moneyness_list,"T":dt_list_curr_tenor, "Spot": spot_list, "Strike":strike_list, "Impl_Vol":impl_vol_list}
        iv_surface.append(vol_smile_dict)
    return iv_surface


# In[ ]:


1. Obj Func -> error * vector () - sum of weights =1 k1, k2, k3 -> c(0.2, 0.4, 0.4)
 a) object func change - loss from squared looss to other loss, abs loss
 b) weights for each strike based on results: (error / total_error) -> each strike
2. Optim algo

---> Saurabh 

3. Parameters in opt - like maxiter, initialisation
4. Combination of optimisation -> saurabh
5. PLot -> Strike vs MV, Heston PV
6. Plot -> Strike vs MKT_IMpl Vol, Heston Impl Vol
7. Plots -> Error func -> gamma, rho, vbar

----> Vishnu

8. Benchmark with Quantlib

----> Abhaya, Vishnu



diff -> error without weights
params

diff -> error with weights
params

basin hopping -> error with weights
params


